# Ночная серия SFT → GRPO на двух T4

Включите Internet, GPU T4 ×2 и доступ к секрету `COMET_API_KEY`. В Add Input прикрепите прежний Spider/BIRD и `sft_full_20261121_103258_artifacts.zip` либо его распакованное содержимое с папкой `evaluation`. Для фонового запуска с чистого ядра выберите Save Version → Save & Run All.

Сначала исходный SFT отвечает на 5819 вопросов train. Из них готовятся две выборки по 768 вопросов. Первая случайная. Во второй до половины составляют вопросы, на которых SQL исходного SFT выполнился и дал ошибочный результат. Фактические доли сохраняются в отчёте.

Для обеих выборок проверяются три комбинации loss и LR. Все шесть обучений стартуют от одного SFT. На каждое выделено 384 шага по два вопроса и четыре ответа на вопрос. После каждой пары валидация проходит на 956 вопросах. Утром сравните парные изменения в selection.json. Все запуски используют seed 42.

In [ ]:
%pip install "torch==2.13.0" "vllm==0.31.0" "transformers==5.17.0" "peft==0.21.2" "accelerate==1.15.0" "comet-ml==3.59.0" python-dotenv
%pip install --no-deps --index-url https://download.pytorch.org/whl/cu130 "torchaudio==2.11.0+cu130"
%pip uninstall -y torchao torchcodec
%pip install --no-deps "trl==1.16.0"
%pip install --no-deps --no-cache-dir --index-url https://download.pytorch.org/whl "nvidia-cudnn-cu13==9.23.0.39"

In [ ]:
import os
import sys
import json
import shutil
import subprocess
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from datetime import datetime, timezone

WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
REPO = WORK / "text2sql-grpo-sft"
RUN = WORK / "text2sql_runs" / "v1"
BASE = "Qwen/Qwen3.5-2B"
REVISION = "15852e8c16360a2fea060d615a32b45270f8a8fc"

if (REPO / ".git").exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/dancravik/text2sql-grpo.git", str(REPO)], check=True)

DATA_DIR = None  # При необходимости укажите Path("/kaggle/input/.../...")
if DATA_DIR is None:
    roots = sorted({
        p.parents[1] for p in INPUT.rglob("train_spider.json")
        if p.parent.name == "spider"
        and (p.parents[1] / "minidev/MINIDEV/mini_dev_sqlite.json").is_file()
    })
    if len(roots) == 1:
        DATA_DIR = roots[0]
    elif roots:
        raise ValueError(f"Укажите DATA_DIR вручную: {roots}")
    else:
        DATA_DIR = WORK / "text2sql_data"
        if not (DATA_DIR / "spider/train_spider.json").is_file():
            archives = list(INPUT.rglob("text2sql-data.zip"))
            if len(archives) != 1:
                raise ValueError(f"Добавьте прежний датасет или задайте DATA_DIR: {archives}")
            with ZipFile(archives[0]) as archive:
                archive.extractall(DATA_DIR)

RUN.mkdir(parents=True, exist_ok=True)
os.environ["DATA_DIR"] = str(DATA_DIR)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ.pop("TORCH_CUDNN_V8_API_DISABLED", None)
subprocess.run([sys.executable, "scripts/freeze_splits.py", "--verify"], cwd=REPO, check=True)
print("DATA_DIR:", DATA_DIR)
print("Свободно GiB:", round(shutil.disk_usage(WORK).free / 1024**3, 2))

In [ ]:
from kaggle_secrets import UserSecretsClient

os.environ["COMET_API_KEY"] = UserSecretsClient().get_secret("COMET_API_KEY")
os.environ["COMET_WORKSPACE"] = "danchenko-rtm"
os.environ["COMET_PROJECT_NAME"] = "text2sql-grpo"
gpu_ids = subprocess.check_output(["nvidia-smi", "--query-gpu=index", "--format=csv,noheader"], text=True).split()
assert len(gpu_ids) == 2, f"Выберите T4 ×2 в настройках. Видны GPU {gpu_ids}"
for gpu in gpu_ids:
    subprocess.run([sys.executable, "-u", "-c", r"""
import importlib.metadata as md
import torch
from transformers import Trainer
from peft import LoraConfig
from vllm import LLM
from trl import GRPOConfig, GRPOTrainer
for pkg, target in {"torch": "2.13.0", "trl": "1.16.0", "vllm": "0.31.0", "nvidia-cudnn-cu13": "9.23.0.39",
                    "transformers": "5.17.0", "peft": "0.21.2", "accelerate": "1.15.0", "comet-ml": "3.59.0"}.items():
    assert md.version(pkg).split("+")[0] == target, (pkg, md.version(pkg), target)
assert torch.cuda.device_count() == 1
x = torch.randn(1, 6144, 512, device="cuda", dtype=torch.float16, requires_grad=True)
w = torch.randn(6144, 1, 4, device="cuda", dtype=torch.float16)
torch.nn.functional.conv1d(x, w, groups=6144, padding=3).float().mean().backward()
torch.cuda.synchronize()
print(torch.cuda.get_device_name(0), "FP16 forward/backward OK")
"""], cwd=REPO, env={**os.environ, "CUDA_VISIBLE_DEVICES": gpu}, check=True)

subprocess.run([sys.executable, "-c", """
from comet_tracking import start_experiment
experiment = start_experiment(name="sft_grpo_overnight_environment")
try:
    experiment.log_metric("environment_ready", 1)
    if not experiment.flush(timeout=120):
        raise RuntimeError("Comet did not confirm the environment check")
finally:
    experiment.end()
"""], cwd=REPO, env={**os.environ, "PYTHONPATH": str(REPO / "src")}, timeout=300, check=True)
print("Обе GPU и запись в Comet проверены.")

Восстановление сохранённого SFT. При нескольких копиях укажите SFT_INPUT как путь к нужной папке в /kaggle/input. Эта ячейка проверяет отчёт и SHA256 весов. Подробная проверка стартовых тензоров выполняется обучающим процессом.

In [ ]:
import hashlib

SFT_INPUT = None  # Например Path("/kaggle/input/datasets/your-name/sft-model")
search_root = Path(SFT_INPUT) if SFT_INPUT is not None else INPUT

def find_sft(root):
    matches = []
    for report_path in root.rglob("run.json"):
        adapter = report_path.parent / "adapter/adapter_model.safetensors"
        if not adapter.is_file() or not (report_path.parent / "evaluation/metrics.json").is_file() or not (report_path.parent / "evaluation/validation.jsonl").is_file():
            continue
        report = json.loads(report_path.read_text())
        if (report.get("status") == "ok" and report.get("num_train_examples") == 5819
                and report.get("global_step") == 728
                and report.get("base_revision") == REVISION):
            matches.append(report_path.parent)
    return sorted(set(matches))

candidates = find_sft(search_root)
if not candidates:
    archives = sorted(search_root.rglob("sft_full_20261121_103258_artifacts.zip"))
    if len(archives) != 1:
        raise ValueError("Добавьте SFT adapter/ и run.json через Add Input. "
                         f"Найдено подходящих ZIP: {archives}")
    restored = WORK / "restored_sft_input"
    restored.mkdir(exist_ok=True)
    with ZipFile(archives[0]) as archive:
        for member in archive.infolist():
            if not (restored / member.filename).resolve().is_relative_to(restored.resolve()):
                raise ValueError("Некорректный путь в ZIP")
        archive.extractall(restored)
    candidates = find_sft(restored)
expected_sha = "8c7ff884c8c80a6345f149c553fee0a24802aab9f079cb7ce67490fb902eb0e4"
candidates = [p for p in candidates if hashlib.sha256((p / "adapter/adapter_model.safetensors").read_bytes()).hexdigest() == expected_sha]
if not candidates:
    raise ValueError(f"Укажите SFT_INPUT с одной копией полного SFT: {candidates}")

SFT = candidates[0]
report = json.loads((SFT / "run.json").read_text())
manifest_hash = hashlib.sha256((REPO / "splits/v1.json").read_bytes()).hexdigest()
assert report["manifest_sha256"] == manifest_hash
assert report["args"]["model"] == BASE and report["optimizer_steps_skipped"] == 0
weights_hash = hashlib.sha256((SFT / "adapter/adapter_model.safetensors").read_bytes()).hexdigest()
print("SFT:", SFT)
print("SHA256 адаптера:", weights_hash)
print("Шагов SFT:", report["global_step"])
if not (SFT / "evaluation/validation.jsonl").is_file():
    raise FileNotFoundError("Входной архив должен содержать evaluation/validation.jsonl для сравнения с SFT")

## Запустить серию

Перед серией оба варианта loss проходят по два тренировочных шага на четырёх длинных промптах. На диске потребуется минимум 11 ГиБ свободного места. Обучения одной пары могут идти параллельно. Слияние весов и оценка выполняются по очереди. Каждый дочерний процесс имеет ограничение времени. Если сеанс прервался и файлы сохранились, повтор этой ячейки найдёт готовые обучения и оценки. Прерванное обучение начинается с исходного SFT. В Comet отправляются снимки весов каждые 128 шагов.

In [ ]:
BUNDLE = RUN / "sft_grpo_overnight_v1"
subprocess.run([
    sys.executable, "-u", "scripts/run_grpo_overnight.py",
    "--out-dir", str(BUNDLE), "--init-sft-run", str(SFT), "--gpus", *gpu_ids, "--steps", "384",
], cwd=REPO, env=os.environ.copy(), check=True)

## Скачать результаты

В ZIP входят исходный и обученные адаптеры, метрики, ответы на каждый вопрос, списки train ID, логи, код и версии пакетов. После завершения фоновой версии архив доступен в Output.

In [ ]:
from IPython.display import FileLink, display

summary = json.loads((BUNDLE / "selection.json").read_text())
print(summary["status"], f"{len(summary['results'])}/6")
for name, row in sorted(summary["results"].items(), key=lambda x: -x[1]["correct"]):
    print(name, f"{row['correct']}/956", f"{row['metrics']['greedy_execution_accuracy']:.2%}",
          "исправлено", row["vs_sft"]["gains"], "потеряно", row["vs_sft"]["losses"])
for stage, error in summary["errors"].items():
    print(stage, error)
archive = BUNDLE.parent / (BUNDLE.name + "_complete.zip")
print(archive)
display(FileLink(archive.relative_to(WORK).as_posix()))

## Дополнительная проверка Pass@4

После всех шести успешных запусков исходный SFT и лучший вариант GRPO получают по четыре попытки на каждом из 200 вопросов validation_fast. При равной основной точности порядок выбора совпадает с порядком волн, внутри волны приоритет у случайной выборки. Для этой проверки выделен остаток девяти часов от старта основной серии. Если осталось меньше часа, ячейка завершится сразу. Результаты сохраняются в отдельный ZIP.

In [ ]:
extra = subprocess.run([
    sys.executable, "-u", "scripts/evaluate_grpo_pass4.py",
    "--study", str(BUNDLE), "--init-sft-run", str(SFT),
    "--gpu", gpu_ids[0], "--hours-from-start", "9",
], cwd=REPO)
if extra.returncode:
    print("Дополнительная оценка прервана. Основной ZIP уже сохранён.")
extra_zip = BUNDLE.parent / (BUNDLE.name + "_pass4.zip")
if extra_zip.is_file():
    display(FileLink(extra_zip.relative_to(WORK).as_posix()))